In [8]:
import os
import glob
import pandas as pd
from autogluon.tabular import TabularPredictor

def find_ag_model_paths(root_dir: str = ".") -> list[str]:
    """
    Finds folders starting with 'ag_models' directly in root_dir (non-recursive).
    """
    return [
        os.path.join(root_dir, name)
        for name in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, name)) and name.startswith("ag_models")
    ]

def export_autogluon_metrics_from_directory(model_paths, output_csv_path: str = "autogluon_all_models_summary.csv") -> pd.DataFrame:
    """
    Finds all 'ag_models*' directories, loads each AutoGluon predictor, 
    extracts validation accuracy, required_features, num_stacks, and ensemble status, 
    and exports everything into a single CSV.
    """
    # model_paths = find_ag_model_paths(root_dir)
    all_extracted_data = []

    for path in model_paths:
        try:
            predictor = TabularPredictor.load(path)
            leaderboard = predictor.leaderboard(extra_info=True, silent=True)
            models_info = predictor.info().get('model_info', {})

            for _, row in leaderboard.iterrows():
                model_name = row['model']
                val_score = row.get('score_val', None)
                num_stack = row.get('stack_level', 0)
                
                model_details = models_info.get(model_name, {})
                required_features = model_details.get('features', predictor.feature_metadata_in.get_features())
                is_ensemble = "WeightedEnsemble" in model_name or model_details.get('model_type') == 'WeightedEnsemble'
                
                all_extracted_data.append({
                    'predictor_path': path,
                    'model_name': model_name,
                    'validation_accuracy': val_score,
                    'num_stacks': num_stack,
                    'is_final_ensembled': is_ensemble,
                    'required_features': list(required_features) if isinstance(required_features, (list, set)) else str(required_features)
                })
        except Exception as e:
            print(f"Skipping {path}: {e}")

    summary_df = pd.DataFrame(all_extracted_data)
    summary_df.to_csv(output_csv_path, index=False)
    
    return summary_df

In [10]:
models= find_ag_model_paths(root_dir="..")

In [11]:
print(models)

['..\\ag_models', '..\\ag_models10', '..\\ag_models11', '..\\ag_models12', '..\\ag_models13', '..\\ag_models14', '..\\ag_models15', '..\\ag_models16', '..\\ag_models17', '..\\ag_models18', '..\\ag_models19', '..\\ag_models2', '..\\ag_models20', '..\\ag_models21', '..\\ag_models22', '..\\ag_models23', '..\\ag_models24', '..\\ag_models25', '..\\ag_models26', '..\\ag_models26_optimize', '..\\ag_models27', '..\\ag_models28', '..\\ag_models29', '..\\ag_models3', '..\\ag_models30', '..\\ag_models31', '..\\ag_models4', '..\\ag_models5', '..\\ag_models6', '..\\ag_models7', '..\\ag_models8', '..\\ag_models9']


In [12]:
models_to_pass = ['..\\ag_models10', '..\\ag_models11', '..\\ag_models12']

In [15]:
results = export_autogluon_metrics_from_directory(models) 

Skipping ..\ag_models: Predictor is not fit. Call `.fit` before calling `.leaderboard`.


This means that the predictor was fit in an AutoGluon version `<=0.3.1`.
This means that the predictor was fit in an AutoGluon version `<=0.3.1`.


Skipping ..\ag_models19: [Errno 2] No such file or directory: 'c:\\Darshak\\Projects\\Hackathon\\ag_models19\\predictor.pkl'
Skipping ..\ag_models2: [Errno 2] No such file or directory: 'c:\\Darshak\\Projects\\Hackathon\\ag_models2\\predictor.pkl'


This means that the predictor was fit in an AutoGluon version `<=0.3.1`.


Skipping ..\ag_models3: [Errno 2] No such file or directory: 'c:\\Darshak\\Projects\\Hackathon\\ag_models3\\predictor.pkl'
Skipping ..\ag_models9: Predictor is not fit. Call `.fit` before calling `.leaderboard`.


In [16]:
results["required_features"]

0      [XGBoost_BAG_L1, LightGBMLarge_BAG_L1, LightGB...
1      [CompoundHardness, StopUrgency, PitWindowStart...
2      [CompoundHardness, StopUrgency, PitWindowStart...
3      [CompoundHardness, StopUrgency, PitWindowStart...
4      [CompoundHardness, StopUrgency, PitWindowStart...
                             ...                        
183    [XGBoost_BAG_L1, LightGBMLarge_BAG_L1, LightGB...
184    [Year, TireRemainingLife, LapsRemaining, PitSt...
185    [Year, TireRemainingLife, LapsRemaining, PitSt...
186    [Year, TireRemainingLife, LapsRemaining, PitSt...
187    [Year, TireRemainingLife, LapsRemaining, PitSt...
Name: required_features, Length: 188, dtype: object

In [60]:
df_results_sampling= pd.read_csv("autogluon_all_models_summary_3.csv")

In [61]:
df_results_sampling[df_results_sampling["is_final_for_model"]==1]

,predictor_path,model_name,validation_accuracy,num_stacks,is_final_ensembled,required_features,data_features,is_final_for_model
0,..\ag_models10,WeightedEnsemble_L2,0.959766,2,True,"['XGBoost_BAG_L1', 'LightGBMLarge_BAG_L1', 'Li...","['CompoundHardness', 'StopUrgency', 'PitWindow...",1.0
5,..\ag_models11,WeightedEnsemble_L2,0.960662,2,True,"['XGBoost_BAG_L1', 'LightGBMLarge_BAG_L1', 'Li...","['Year', 'TireRemainingLife', 'LapsRemaining',...",1.0
10,..\ag_models12,WeightedEnsemble_L2,0.960410,2,True,"['XGBoost_BAG_L1', 'LightGBMLarge_BAG_L1', 'Ca...","['Year', 'TireRemainingLife', 'LapsRemaining',...",1.0
15,..\ag_models13,WeightedEnsemble_L2,0.958999,2,True,"['XGBoost_BAG_L1', 'LightGBMLarge_BAG_L1', 'Li...","['Year', 'TireRemainingLife', 'LapsRemaining',...",1.0
20,..\ag_models14,WeightedEnsemble_L3,0.963153,3,True,"['CatBoost_BAG_L2', 'NeuralNetTorch_BAG_L2', '...","['Year', 'TireRemainingLife', 'LapsRemaining',...",1.0
34,..\ag_models17,WeightedEnsemble_L3,0.961982,3,True,"['CatBoost_BAG_L2', 'LightGBMLarge_BAG_L2', 'X...","['Year', 'TireRemainingLife', 'LapsRemaining',...",1.0
44,..\ag_models18,WeightedEnsemble_L3,0.962055,3,True,"['CatBoost_BAG_L2', 'LightGBMLarge_BAG_L2', 'X...","['Year', 'TireRemainingLife', 'LapsRemaining',...",1.0
54,..\ag_models20,WeightedEnsemble_L3,0.959648,3,True,"['XGBoost_BAG_L2', 'LightGBMLarge_BAG_L2', 'Ne...","['Year', 'TireRemainingLife', 'LapsRemaining',...",1.0
68,..\ag_models21,WeightedEnsemble_L2,0.960061,2,True,"['XGBoost_BAG_L1', 'LightGBMLarge_BAG_L1', 'Ca...","['CompoundTyreLifeRemainingPct', 'Year', 'Tire...",1.0
73,..\ag_models22,WeightedEnsemble_L2,0.960633,2,True,"['XGBoost_BAG_L1', 'LightGBMLarge_BAG_L1', 'Ca...","['Stint_Tyre_Min_Life', 'Year', 'Stint_Tyre_Me...",1.0


In [62]:
df_results_sampling["data_features"].value_counts()

data_features
['Year', 'TireRemainingLife', 'LapsRemaining', 'Driver', 'PitStop', 'Prev_Position', 'LapNumber', 'RaceLaps', 'Compound', 'RaceProgress', 'TireAgePct', 'MaxTireLife', 'Race', 'Stint', 'TyreLife', 'TireRemainingPct', 'LapTime (s)', 'LapTimeAvg_3', 'LapTime_Delta', 'Position', 'Position_Change', 'Cumulative_Degradation']                                                                                                                                                                                                                                                                                                                                                                                 6
['Stint_Tyre_Min_Life', 'Year', 'Stint_Tyre_Mean_Life', 'TireRemainingLife', 'Stint_Tyre_Median_Life', 'LapsRemaining', 'PitStop', 'Prev_Position', 'LapNumber', 'RaceLaps', 'Compound', 'RaceProgress', 'TireAgePct', 'MaxTireLife', 'Stint_Tyre_Std_Dev', 'Race', 'Stint', 'TyreLife', 'TireRemaining

In [63]:
df_results_sampling["data_features"].unique()

array(["['CompoundHardness', 'StopUrgency', 'PitWindowStartPct', 'Year', 'IsDegrading', 'TyreAge_x_RaceProgress', 'TyreLifeMinusExpectedWindow', 'AbsLapTimeDelta', 'LapsRemaining', 'Driver', 'PitStop', 'Prev_Position', 'LapNumber', 'RaceLaps', 'Compound', 'AbsPositionChange', 'WindowOvershootPct', 'PositionLoss', 'StintCapped', 'RaceProgress', 'ExpectedMaxLife', 'TireAgePct', 'DegPerLap', 'Race', 'Stint', 'TyreLife', 'TireRemainingPct', 'LapTime (s)', 'TyreLife_x_CompoundHardness', 'RaceProgress_x_Stint', 'PositiveLapTimeDelta', 'TyreAge_x_LapTimeDelta', 'LapTimeAvg_3', 'LapTime_Delta', 'InCompoundPitWindow', 'RaceRemainingPct', 'Position', 'Position_Change', 'Cumulative_Degradation']",
       nan,
       "['Year', 'TireRemainingLife', 'LapsRemaining', 'Driver', 'PitStop', 'Prev_Position', 'LapNumber', 'RaceLaps', 'Compound', 'RaceProgress', 'TireAgePct', 'MaxTireLife', 'Race', 'Stint', 'TyreLife', 'TireRemainingPct', 'LapTime (s)', 'LapTimeAvg_3', 'LapTime_Delta', 'Position', 'Positio

In [64]:
base = ['PitStop', 'TyreLife', 'Stint', 'LapTime_Delta', 'Race', 'LapNumber', 'LapTime (s)', 'Compound', 'Year', 'RaceProgress', 'Position', 'Position_Change', 'Cumulative_Degradation'] 
base_without_driver = ['PitStop', 'Stint', 'TyreLife', 'LapTime_Delta', 'id', 'Driver', 'LapNumber', 'Race', 'LapTime (s)', 'Compound', 'Year', 'RaceProgress', 'Position', 'Position_Change', 'Cumulative_Degradation']
fe_v1_with_driver = ['Year', 'TireRemainingLife', 'LapsRemaining', 'Driver', 'PitStop', 'Prev_Position', 'LapNumber', 'RaceLaps', 'Compound', 'RaceProgress', 'TireAgePct', 'MaxTireLife', 'Race', 'Stint', 'TyreLife', 'TireRemainingPct', 'LapTime (s)', 'LapTimeAvg_3', 'LapTime_Delta', 'Position', 'Position_Change', 'Cumulative_Degradation']
fe_v1_without_driver = ['Year', 'TireRemainingLife', 'LapsRemaining', 'PitStop', 'Prev_Position', 'LapNumber', 'RaceLaps', 'Compound', 'RaceProgress', 'TireAgePct', 'MaxTireLife', 'Race', 'Stint', 'TyreLife', 'TireRemainingPct', 'LapTime (s)', 'LapTimeAvg_3', 'LapTime_Delta', 'Position', 'Position_Change', 'Cumulative_Degradation']
fe_v4_wih_driver = ['Year', 'TireRemainingLife', 'LapsRemaining', 'Driver', 'PitStop', 'Prev_Position', 'LapNumber', 'RaceLaps', 'Compound', 'RaceProgress', 'TireAgePct', 'MaxTireLife', 'Race', 'Stint', 'TyreLife', 'TireRemainingPct', 'LapTime (s)', 'LapTimeAvg_3', 'LapTime_Delta', 'Normalized_TyreLife', 'Position', 'Position_Change', 'Cumulative_Degradation']
fe_v6_without_driver =['Stint_Tyre_Min_Life', 'Year', 'Stint_Tyre_Mean_Life', 'TireRemainingLife', 'Stint_Tyre_Median_Life', 'LapsRemaining', 'PitStop', 'Prev_Position', 'LapNumber', 'RaceLaps', 'Compound', 'RaceProgress', 'TireAgePct', 'MaxTireLife', 'Stint_Tyre_Std_Dev', 'Race', 'Stint', 'TyreLife', 'TireRemainingPct', 'Stint_Tyre_Max_Life', 'LapTime (s)', 'LapTimeAvg_3', 'LapTime_Delta', 'Stint_Count', 'Position', 'Position_Change', 'Cumulative_Degradation'] 
fe_v6_with_driver = ['Stint_Tyre_Min_Life', 'Year', 'Stint_Tyre_Mean_Life', 'TireRemainingLife', 'Stint_Tyre_Median_Life', 'LapsRemaining', 'Driver', 'PitStop', 'Prev_Position', 'LapNumber', 'RaceLaps', 'Compound', 'RaceProgress', 'TireAgePct', 'MaxTireLife', 'Stint_Tyre_Std_Dev', 'Race', 'Stint', 'TyreLife', 'TireRemainingPct', 'Stint_Tyre_Max_Life', 'LapTime (s)', 'LapTimeAvg_3', 'LapTime_Delta', 'Stint_Count', 'Position', 'Position_Change', 'Cumulative_Degradation']
fe_v7_with_driver = ['Stint_Tyre_Min_Life', 'Race_Lap_Min', 'Year', 'Stint_Tyre_Mean_Life', 'TireRemainingLife', 'Stint_Tyre_Median_Life', 'LapsRemaining', 'Driver', 'PitStop', 'Race_Lap_Mean', 'Prev_Position', 'LapNumber', 'RaceLaps', 'Compound', 'RaceProgress', 'Race_Lap_Max', 'TireAgePct', 'MaxTireLife', 'Stint_Tyre_Std_Dev', 'Race', 'Stint', 'TyreLife', 'TireRemainingPct', 'Stint_Tyre_Max_Life', 'LapTime (s)', 'Race_Lap_Median', 'Race_Lap_Count', 'Race_Lap_Std_Dev', 'LapTimeAvg_3', 'LapTime_Delta', 'Stint_Count', 'Position', 'Position_Change', 'Cumulative_Degradation']
fe_v2_without_driver = ['CompoundHardness', 'StopUrgency', 'PitWindowStartPct', 'Year', 'IsDegrading', 'TyreAge_x_RaceProgress', 'TyreLifeMinusExpectedWindow', 'AbsLapTimeDelta', 'LapsRemaining', 'Driver', 'PitStop', 'Prev_Position', 'LapNumber', 'RaceLaps', 'Compound', 'AbsPositionChange', 'WindowOvershootPct', 'PositionLoss', 'StintCapped', 'RaceProgress', 'ExpectedMaxLife', 'TireAgePct', 'DegPerLap', 'Race', 'Stint', 'TyreLife', 'TireRemainingPct', 'LapTime (s)', 'TyreLife_x_CompoundHardness', 'RaceProgress_x_Stint', 'PositiveLapTimeDelta', 'TyreAge_x_LapTimeDelta', 'LapTimeAvg_3', 'LapTime_Delta', 'InCompoundPitWindow', 'RaceRemainingPct', 'Position', 'Position_Change', 'Cumulative_Degradation']
fe_v3_with_driver = ['CompoundTyreLifeRemainingPct', 'Year', 'TireRemainingLife', 'LapsRemaining', 'Driver', 'PitStop', 'Prev_Position', 'LapNumber', 'RaceLaps', 'Compound', 'RaceProgress', 'TireAgePct', 'MaxTireLife', 'Race', 'Stint', 'TyreLife', 'TireRemainingPct', 'LapTime (s)', 'CompoundTyreLifeUsedPct', 'LapTimeAvg_3', 'LapTime_Delta', 'CompoundMeanTyreLife', 'Position', 'CompoundTyreLifeRemaining', 'Position_Change', 'Cumulative_Degradation']
fe_v8_without_driver =['Stint_Tyre_Min_Life', 'Year', 'Stint_Tyre_Mean_Life', 'TireRemainingLife', 'Stint_Tyre_Median_Life', 'LapsRemaining', 'PitStop', 'Prev_Position', 'LapNumber', 'RaceLaps', 'Compound', 'Compound_Hardness', 'RaceProgress', 'PitCount', 'TireAgePct', 'MaxTireLife', 'Stint_Tyre_Std_Dev', 'Race', 'Stint', 'TyreLife', 'TireRemainingPct', 'Stint_Tyre_Max_Life', 'LapTime (s)', 'LapTimeAvg_3', 'LapTime_Delta', 'Stint_Count', 'Position_Change_3Lap', 'RollingLapTimeDelta_3', 'Position', 'Position_Change', 'Cumulative_Degradation']
fe_v8_without_driver_external_data_validation = ['Stint_Tyre_Min_Life', 'Year', 'Stint_Tyre_Mean_Life', 'TireRemainingLife', 'Stint_Tyre_Median_Life', 'LapsRemaining', 'PitStop', 'Prev_Position', 'LapNumber', 'RaceLaps', 'Compound', 'Compound_Hardness', 'RaceProgress', 'PitCount', 'TireAgePct', 'MaxTireLife', 'Stint_Tyre_Std_Dev', 'Race', 'Stint', 'TyreLife', 'TireRemainingPct', 'Stint_Tyre_Max_Life', 'LapTime (s)', 'LapTimeAvg_3', 'is_external_data', 'LapTime_Delta', 'Stint_Count', 'Position_Change_3Lap', 'RollingLapTimeDelta_3', 'Position', 'Position_Change', 'Cumulative_Degradation'] 


In [65]:
df_results_sampling["data_features"]

0      ['CompoundHardness', 'StopUrgency', 'PitWindow...
1                                                    NaN
2                                                    NaN
3                                                    NaN
4                                                    NaN
                             ...                        
183    ['Year', 'TireRemainingLife', 'LapsRemaining',...
184                                                  NaN
185                                                  NaN
186                                                  NaN
187                                                  NaN
Name: data_features, Length: 188, dtype: object

In [66]:
df_results_sampling["data_features"] = (
    df_results_sampling["data_features"].fillna("NO_FINAL")
)

In [67]:
df_results_sampling["data_features"]

0      ['CompoundHardness', 'StopUrgency', 'PitWindow...
1                                               NO_FINAL
2                                               NO_FINAL
3                                               NO_FINAL
4                                               NO_FINAL
                             ...                        
183    ['Year', 'TireRemainingLife', 'LapsRemaining',...
184                                             NO_FINAL
185                                             NO_FINAL
186                                             NO_FINAL
187                                             NO_FINAL
Name: data_features, Length: 188, dtype: object

In [ ]:
# Feature sets
feature_versions = {
    'base': base,
    'base_without_driver': base_without_driver,
    'fe_v1_with_driver': fe_v1_with_driver,
    'fe_v1_without_driver': fe_v1_without_driver,
    'fe_v2_without_driver': fe_v2_without_driver,
    'fe_v3_with_driver': fe_v3_with_driver,
    'fe_v4_wih_driver': fe_v4_wih_driver,
    'fe_v6_without_driver': fe_v6_without_driver,
    'fe_v6_with_driver': fe_v6_with_driver,
    'fe_v7_with_driver': fe_v7_with_driver,
    'fe_v8_without_driver': fe_v8_without_driver,
    'fe_v8_without_driver_external_data_validation': fe_v8_without_driver_external_data_validation
}



def get_fe_version(features):
    if features not in ["","[]","NO_FINAL"]:
        print(features)
        # feature_set = set(features)

        for version, version_features in feature_versions.items():
            print(type(feature_set),type(version_features))
            if feature_set == version_features:
                return version

        return 'fe_multi_stack'




df_results_sampling['fe_version'] = (
    df_results_sampling['data_features']
    .apply(get_fe_version)
)

['CompoundHardness', 'StopUrgency', 'PitWindowStartPct', 'Year', 'IsDegrading', 'TyreAge_x_RaceProgress', 'TyreLifeMinusExpectedWindow', 'AbsLapTimeDelta', 'LapsRemaining', 'Driver', 'PitStop', 'Prev_Position', 'LapNumber', 'RaceLaps', 'Compound', 'AbsPositionChange', 'WindowOvershootPct', 'PositionLoss', 'StintCapped', 'RaceProgress', 'ExpectedMaxLife', 'TireAgePct', 'DegPerLap', 'Race', 'Stint', 'TyreLife', 'TireRemainingPct', 'LapTime (s)', 'TyreLife_x_CompoundHardness', 'RaceProgress_x_Stint', 'PositiveLapTimeDelta', 'TyreAge_x_LapTimeDelta', 'LapTimeAvg_3', 'LapTime_Delta', 'InCompoundPitWindow', 'RaceRemainingPct', 'Position', 'Position_Change', 'Cumulative_Degradation']
<class 'str'> <class 'list'>
<class 'str'> <class 'list'>
<class 'str'> <class 'list'>
<class 'str'> <class 'list'>
<class 'str'> <class 'list'>
<class 'str'> <class 'list'>
<class 'str'> <class 'list'>
<class 'str'> <class 'list'>
<class 'str'> <class 'list'>
<class 'str'> <class 'list'>
<class 'str'> <class 'li

In [75]:
df_results_sampling['fe_version'].value_counts()

fe_version
fe_multi_stack    24
Name: count, dtype: int64